# Multimodal Extraction on Colab T4

Notebook này chạy test 1 frame trước, sau đó chạy batch theo từng file week*_chunks.json.

## Cell 1: Cài dependencies

In [1]:
!pip -q install -U --force-reinstall --no-cache-dir 'pillow<12'
!pip -q install -U transformers accelerate bitsandbytes qwen-vl-utils tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 184.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 134.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 37.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 69.6 MB/s eta 0:00:00:00:0100:01


In [2]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


## Cell 2: Mount Google Drive và cấu hình path

In [3]:


# TODO: đổi path này theo thư mục bạn upload
PROJECT_ROOT = Path('/content/drive/MyDrive')
TEXT_CHUNKS_DIR = PROJECT_ROOT / 'data/interim/text_chunks'
FRAMES_DIR = PROJECT_ROOT / 'data/interim/frames'
SYSTEM_PROMPT_PATH = PROJECT_ROOT / 'data/system_prompt.txt'
OUTPUT_DIR = PROJECT_ROOT / 'data/processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('TEXT_CHUNKS_DIR exists:', TEXT_CHUNKS_DIR.exists())
print('FRAMES_DIR exists:', FRAMES_DIR.exists())
print('SYSTEM_PROMPT_PATH exists:', SYSTEM_PROMPT_PATH.exists())

PROJECT_ROOT = /content/drive/MyDrive
TEXT_CHUNKS_DIR exists: True
FRAMES_DIR exists: True
SYSTEM_PROMPT_PATH exists: True


## Cell 3: Load model Qwen2.5-VL-3B

In [4]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto'
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

print('Model loaded:', MODEL_ID)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model loaded: Qwen/Qwen2.5-VL-3B-Instruct


## Cell 4: Helper functions

In [5]:
import json
import re
from typing import Any
from qwen_vl_utils import process_vision_info

with SYSTEM_PROMPT_PATH.open('r', encoding='utf-8') as f:
    SYSTEM_PROMPT = f.read().strip()

def extract_json_text(text: str) -> str:
    s = text.strip()
    if s.startswith('```'):
        s = re.sub(r'^```(?:json)?\s*', '', s, flags=re.IGNORECASE)
        s = re.sub(r'\s*```$', '', s)
    first = s.find('{')
    last = s.rfind('}')
    if first != -1 and last != -1 and last > first:
        return s[first:last+1]
    return s

def run_vlm_on_image(image_path: Path, user_instruction: str, max_new_tokens: int = 256) -> dict[str, Any]:
    messages = [
        {
            'role': 'system',
            'content': [{'type': 'text', 'text': SYSTEM_PROMPT}]
        },
        {
            'role': 'user',
            'content': [
                {'type': 'image', 'image': str(image_path)},
                {'type': 'text', 'text': user_instruction}
            ]
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors='pt'
    ).to(model.device)

    try:
        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                num_beams=1
            )
    except Exception as exc:
        raise RuntimeError(f'Generation failed for image {image_path.name}: {exc}') from exc

    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    raw = output_text.strip()
    try:
        parsed = json.loads(extract_json_text(raw))
    except Exception:
        parsed = {
            'is_valid': False,
            'ocr_text': '',
            'visual_description': '',
            'raw_response': raw
        }

    for key in ['is_valid', 'ocr_text', 'visual_description']:
        if key not in parsed:
            parsed[key] = False if key == 'is_valid' else ''

    return parsed

## Cell 5: Test 1 frame trước

In [6]:
test_frame = sorted(FRAMES_DIR.glob('*.jpg'))[0]
print('Testing frame:', test_frame.name)

result = run_vlm_on_image(
    test_frame,
    'Analyze this keyframe and return strict JSON with keys is_valid, ocr_text, visual_description.'
)

print(json.dumps(result, ensure_ascii=False, indent=2))

Testing frame: SlqjA04_dpk_0020.jpg


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


{
  "is_valid": true,
  "ocr_text": "",
  "visual_description": "The instructor is standing in front of a large screen displaying a diagram with a purple rectangle labeled 'play' connected to a cat icon labeled 'play music'. The instructor appears to be explaining something related to the diagram."
}


In [7]:
from tqdm import tqdm

## Cell 6: Chạy 1 file week cụ thể (khuyên dùng trước)

In [ ]:

WEEK_FILE = TEXT_CHUNKS_DIR / 'week9_flask_chunks.json'
LIMIT = None  # đổi thành số nhỏ như 10 để test nhanh

with WEEK_FILE.open('r', encoding='utf-8') as f:
    chunks = json.load(f)

if LIMIT is not None:
    chunks = chunks[:LIMIT]

processed = []
cuda_assert_hit = False
for ch in tqdm(chunks, desc=f'Processing {WEEK_FILE.name}'):
    chunk_id = str(ch.get('chunk_id', '')).strip()
    frame_path = FRAMES_DIR / f'{chunk_id}.jpg'

    out = dict(ch)
    if not frame_path.exists():
        out['ocr_text'] = out.get('ocr_text', '')
        out['visual_description'] = out.get('visual_description', '')
        processed.append(out)
        continue

    try:
        resp = run_vlm_on_image(
            frame_path,
            f'chunk_id={chunk_id}. Return strict JSON only.',
            max_new_tokens=256
        )
    except Exception as exc:
        err = str(exc)
        out['ocr_text'] = ''
        out['visual_description'] = ''
        out['error'] = err
        processed.append(out)
        print(f'[WARN] chunk_id={chunk_id} failed: {err[:180]}')
        if 'device-side assert' in err.lower() or 'cuda error' in err.lower():
            cuda_assert_hit = True
            print('[FATAL] CUDA assert detected. Saving partial output now.')
            break
        continue

    if bool(resp.get('is_valid', False)):
        out['ocr_text'] = str(resp.get('ocr_text', '')).strip()
        out['visual_description'] = str(resp.get('visual_description', '')).strip()
    else:
        out['ocr_text'] = ''
        out['visual_description'] = ''

    processed.append(out)

out_path = OUTPUT_DIR / WEEK_FILE.name
with out_path.open('w', encoding='utf-8') as f:
    json.dump(processed, f, ensure_ascii=False, indent=2)

print('Saved:', out_path)
print('Rows:', len(processed))
if cuda_assert_hit:
    print('Runtime GPU context may be broken. Please Runtime -> Restart runtime, then rerun from Cell 3.')  

Processing week9_flask_chunks.json: 100%|██████████| 160/160 [20:31<00:00,  7.70s/it]

Saved: /content/drive/MyDrive/data/processed/week9_flask_chunks.json
Rows: 160
